In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
CSV_DIR = Path("csv")
OUT_DIR = Path("paper_figures")
OUT_DIR.mkdir(exist_ok=True)

# Update these names if needed.
SUCCESS_CSV = CSV_DIR / "wandb_export_2026-07-08T14_40_56.879-04_00.csv"
REWARD_CSV  = CSV_DIR / "wandb_export_2026-07-08T14_40_46.666-04_00.csv"

# Set to 1 to avoid extra smoothing.
# Your W&B metrics are already rolling metrics from the training callback.
EXTRA_ROLLING_WINDOW = 1

In [ ]:
def find_x_column(df: pd.DataFrame) -> str:
    if "train/episode" in df.columns:
        return "train/episode"
    if "Step" in df.columns:
        return "Step"
    raise ValueError("Could not find x-axis column. Expected 'train/episode' or 'Step'.")


def find_run_metric_columns(df: pd.DataFrame, metric_name: str):
    """
    Finds columns like:
        helpful-fog-1823 - train/success_rate
        ppo-seed-sweep-19 - train/reward_mean

    Ignores W&B helper columns:
        __MIN, __MAX, _step
    """
    metric_suffix = f"train/{metric_name}"
    cols = []

    for col in df.columns:
        if "__MIN" in col or "__MAX" in col or " - _step" in col:
            continue
        if col.endswith(metric_suffix) and " - " in col:
            cols.append(col)

    if not cols:
        raise ValueError(f"No run columns found for metric: {metric_name}")

    return cols


def load_ppo_metric(path: Path, metric_name: str) -> pd.DataFrame:
    """
    Converts a W&B wide CSV export into an aggregate table:

        episode, mean, min, max, n_runs

    The min/max are computed across PPO runs at each episode.
    """
    df = pd.read_csv(path)
    x_col = find_x_column(df)
    metric_cols = find_run_metric_columns(df, metric_name)

    work = df[[x_col] + metric_cols].copy()
    work = work.rename(columns={x_col: "episode"})

    work["episode"] = pd.to_numeric(work["episode"], errors="coerce")
    for col in metric_cols:
        work[col] = pd.to_numeric(work[col], errors="coerce")

    work = work.dropna(subset=["episode"])
    work["episode"] = work["episode"].astype(int)

    values = work[metric_cols]

    agg = pd.DataFrame({
        "episode": work["episode"],
        "mean": values.mean(axis=1, skipna=True),
        "min": values.min(axis=1, skipna=True),
        "max": values.max(axis=1, skipna=True),
        "n_runs": values.notna().sum(axis=1),
    })

    if EXTRA_ROLLING_WINDOW > 1:
        for col in ["mean", "min", "max"]:
            agg[col] = agg[col].rolling(EXTRA_ROLLING_WINDOW, min_periods=1).mean()

    return agg, metric_cols


success_df, success_cols = load_ppo_metric(SUCCESS_CSV, "success_rate")
reward_df, reward_cols = load_ppo_metric(REWARD_CSV, "reward_mean")

print(f"Success-rate runs detected: {len(success_cols)}")
print(f"Reward-mean runs detected: {len(reward_cols)}")

display(success_df.tail())
display(reward_df.tail())

In [ ]:
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "lines.linewidth": 2.6,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def plot_mean_minmax(ax, df, ylabel, panel_title, ylim=None):
    ax.fill_between(
        df["episode"],
        df["min"],
        df["max"],
        alpha=0.20,
        label="Min--Max Range",
        linewidth=0,
    )

    ax.plot(
        df["episode"],
        df["mean"],
        label="Mean",
        linewidth=2.8,
        zorder=3,
    )

    ax.set_xlabel("Training episode")
    ax.set_ylabel(ylabel)
    ax.set_title(panel_title, loc="left", fontweight="bold")

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.tick_params(axis="both", which="major")


fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.5, 4.6),
    constrained_layout=True,
)

plot_mean_minmax(
    axes[0],
    success_df,
    ylabel="Success rate",
    panel_title="(a) PPO success rate across seeds",
    ylim=(-0.05, 1.05),
)

axes[0].set_yticks(np.linspace(0, 1, 6))

plot_mean_minmax(
    axes[1],
    reward_df,
    ylabel="Mean episodic return",
    panel_title="(b) PPO mean episodic return across seeds",
)

# One shared legend.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.08),
)

fig.suptitle(
    "PPO seed variability",
    y=1.14,
    fontsize=14,
    fontweight="bold",
)

fig.savefig(OUT_DIR / "fig_ppo_seed_variability.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / "fig_ppo_seed_variability.png", bbox_inches="tight", dpi=600)

plt.show()

In [ ]:
success_df.to_csv(OUT_DIR / "ppo_success_rate_mean_minmax.csv", index=False)
reward_df.to_csv(OUT_DIR / "ppo_reward_mean_minmax.csv", index=False)